In [ ]:

import math
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

tf.keras.backend.clear_session()
tf.random.set_seed(0)
np.random.seed(0)


In [ ]:
import tensorflow as tf
#tf.keras.mixed_precision.set_global_policy('mixed_float16')

import tensorflow as tf
import numpy as np
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet121, ResNet50V2
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet169, MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy

import tensorflow as tf
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras import layers

### Input Multimodal data

In [ ]:
X_train_image = np.load('X_train1.npy')
X_train_ehr = np.load('X_train2.npy')
y_train = np.load('y_train.npy')

X_val_image = np.load('X_val1.npy')
X_val_ehr = np.load('X_val2.npy')
y_val = np.load('y_val.npy')

X_test_image = np.load('X_test1.npy')
X_test_ehr = np.load('X_test2.npy')
y_test = np.load('y_test.npy')

X_train_image.shape, X_train_ehr.shape, y_train.shape, X_val_image.shape, X_val_ehr.shape, y_val.shape, X_test_image.shape, X_test_ehr.shape, y_test.shape

In [ ]:

# ============================================================
# SWAN
# ============================================================

class GlobalAttentionNoiseLayer(layers.Layer):
    def __init__(self, noise_factor=0.2, **kwargs):
        super(GlobalAttentionNoiseLayer, self).__init__(**kwargs)
        self.noise_factor = noise_factor

    def build(self, input_shape):
        self.alpha = self.add_weight(
            shape=(1,),
            initializer='ones',
            trainable=True,
            #name='alpha'
        )
        super(GlobalAttentionNoiseLayer, self).build(input_shape)

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(
                shape=tf.shape(inputs),
                mean=0.0,
                stddev=self.noise_factor * self.alpha
            )
            return inputs + noise
        else:
            return inputs


class FeatureAttentionNoiseLayer(layers.Layer):
    def __init__(self, noise_factor=0.2, learning_rate=0.001, **kwargs):
        super(FeatureAttentionNoiseLayer, self).__init__(**kwargs)
        self.noise_factor = noise_factor
        self.learning_rate = learning_rate

    def build(self, input_shape):
        _, H, W, C = input_shape
        self.alpha = self.add_weight(
            shape=(H, W, C),
            initializer='ones',
            trainable=True,
            #name='alpha'
        )
        self.dense = layers.Dense(units=C, activation='relu')
        super(FeatureAttentionNoiseLayer, self).build(input_shape)

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(shape=tf.shape(inputs),
                                     mean=0.0,
                                     stddev=self.noise_factor)

            global_avg_pooled = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
            delta_related = self.dense(global_avg_pooled)

            attention_noise = inputs + delta_related * inputs * self.alpha * noise
            return attention_noise
        else:
            return inputs

    def get_config(self):
        config = super(FeatureAttentionNoiseLayer, self).get_config()
        config.update({
            'noise_factor': self.noise_factor,
            'learning_rate': self.learning_rate
        })
        return config


class CombinedNoiseLayer(layers.Layer):
    def __init__(self, global_noise_factor=0.2, feature_noise_factor=0.2,
                 learning_rate=0.001, **kwargs):
        super(CombinedNoiseLayer, self).__init__(**kwargs)
        self.global_noise_layer = GlobalAttentionNoiseLayer(
            noise_factor=global_noise_factor
        )
        self.feature_noise_layer = FeatureAttentionNoiseLayer(
            noise_factor=feature_noise_factor,
            learning_rate=learning_rate
        )

    def call(self, inputs, training=None):
        if training:
            x = self.global_noise_layer(inputs, training=training)
            x = self.feature_noise_layer(x, training=training)
            return x
        else:
            return inputs

    def get_config(self):
        config = super(CombinedNoiseLayer, self).get_config()
        config.update({
            'global_noise_factor': self.global_noise_layer.noise_factor,
            'feature_noise_factor': self.feature_noise_layer.noise_factor,
            'learning_rate': self.feature_noise_layer.learning_rate
        })
        return config


class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(
            filters=1,
            kernel_size=(1, 1),
            activation='sigmoid',
            padding='same',
            trainable=False
        )
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)


class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 8,
                            activation='relu',
                            trainable=False)
        self.dense2 = Dense(units=input_shape[-1],
                            activation='sigmoid',
                            trainable=False)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(
            tf.expand_dims(channel_attention_weights, 1), 1
        )
        return tf.multiply(inputs, channel_attention_weights)


class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=0.6,
                 channel_noise_factor=0.6, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [
            TrainableSpatialAttentionLayer() for _ in range(num_layers)
        ]
        self.channel_attentions = [
            TrainableChannelAttentionLayer() for _ in range(num_layers)
        ]

        self.spatial_noise_weight = self.add_weight(
            shape=(1,),
            initializer='ones',
            trainable=False,
            #name='spatial_noise_weight'
        )
        self.channel_noise_weight = self.add_weight(
            shape=(1,),
            initializer='ones',
            trainable=False,
            #name='channel_noise_weight'
        )

        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(
            self.spatial_attentions, self.channel_attentions
        ):
            spatial_attention_output = spatial_attention(spatial_attention_output)
            channel_attention_output = channel_attention(channel_attention_output)

        spatial_noise = tf.random.normal(
            shape=tf.shape(spatial_attention_output),
            mean=0,
            stddev=self.spatial_noise_factor
        )
        channel_noise = tf.random.normal(
            shape=tf.shape(channel_attention_output),
            mean=0,
            stddev=self.channel_noise_factor
        )

        spatial_noise *= self.spatial_noise_weight + (
            spatial_noise * self.spatial_noise_weight
        )
        channel_noise *= self.channel_noise_weight + (
            channel_noise * self.channel_noise_weight
        )

        spatial_attention_output += spatial_noise
        channel_attention_output += channel_noise

        spatial_attention_output = tf.clip_by_value(
            spatial_attention_output, 0.0, 1.0
        )
        channel_attention_output = tf.clip_by_value(
            channel_attention_output, 0.0, 1.0
        )

        combined_attention1 = tf.multiply(
            spatial_attention_output, channel_attention_output
        )
        combined_attention2 = tf.multiply(
            spatial_attention_output, channel_attention_output
        )
        combined_attention3 = tf.multiply(
            spatial_attention_output, channel_attention_output
        )
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)

        return tf.multiply(inputs, combined_attention)


class SoftplusMergedAttentionNoiseLayer(layers.Layer):
    def __init__(self,
                 global_noise_factor=0.2,
                 feature_noise_factor=0.2,
                 learning_rate=0.001,
                 spatial_noise_factor=0.6,
                 channel_noise_factor=0.6,
                 num_layers=3,
                 use_eval_noise=True,
                 eps=1e-6,
                 **kwargs):
        super(SoftplusMergedAttentionNoiseLayer, self).__init__(**kwargs)

        self.combined_noise = CombinedNoiseLayer(
            global_noise_factor=global_noise_factor,
            feature_noise_factor=feature_noise_factor,
            learning_rate=learning_rate
        )

        '''self.attention_noise = TrainableCombinedAttentionLayer(
            spatial_noise_factor=spatial_noise_factor,
            channel_noise_factor=channel_noise_factor,
            num_layers=num_layers
        )'''

        self.use_eval_noise = bool(use_eval_noise)
        self.eps = float(eps)

    def build(self, input_shape):
        self.theta_global_feat = self.add_weight(
            #name=self.name + "_theta_global_feat",
            shape=(),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )
        self.theta_att = self.add_weight(
            #name=self.name + "_theta_att",
            shape=(),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs, training=None):
        x = inputs

        if training:
            x_gf = self.combined_noise(x, training=True)
        else:
            if self.use_eval_noise:
                x_gf = self.combined_noise(x, training=True)
            else:
                x_gf = self.combined_noise(x, training=False)

        #x_att = self.attention_noise(x)

        delta_gf = x_gf - x
        delta_att = x_gf + x

        w_g = tf.nn.softplus(self.theta_global_feat)
        w_a = tf.nn.softplus(self.theta_att)

        denom = w_g + w_a + self.eps
        delta = (w_g * delta_gf + w_a * delta_att) / denom

        y = x + delta
        return y


In [ ]:
# ============================================================
# 2. Utilities
# ============================================================

def spectral_clip_flat_kernel(kernel, sigma_max=None, eps=1e-6):
    """
    Proxy spectral clipping:
    flatten [kh, kw, Cin, Cout] -> [kh*kw*Cin, Cout],
    clip the matrix spectral norm to sigma_max.

    This is a practical proxy, not the exact convolution-operator norm.
    """
    if sigma_max is None:
        return kernel

    kernel = tf.convert_to_tensor(kernel)
    cout = tf.shape(kernel)[-1]
    flat = tf.reshape(kernel, [-1, cout])
    singular_values = tf.linalg.svd(flat, compute_uv=False)
    sigma = singular_values[0]
    sigma_max = tf.cast(sigma_max, kernel.dtype)
    scale = tf.minimum(
        tf.cast(1.0, kernel.dtype),
        sigma_max / (sigma + tf.cast(eps, kernel.dtype))
    )
    return kernel * scale


def _iter_nested_keras_layers(root):
    """
    Keras-3-safe recursive layer traversal.

    Why this helper exists:
      - Functional models expose `model.layers`, not `model.submodules`.
      - Custom blocks often subclass `layers.Layer`, and store child layers
        as Python attributes.

    So we recursively walk:
      1) the public `.layers` list when present
      2) layer-valued attributes on custom layers
      3) containers holding Layer objects
    """
    seen = set()

    def visit(obj):
        if isinstance(obj, layers.Layer):
            obj_id = id(obj)
            if obj_id in seen:
                return
            seen.add(obj_id)

            yield obj

            # Public container path
            child_layers = getattr(obj, "layers", None)
            if child_layers is not None:
                for child in child_layers:
                    yield from visit(child)

            # Fallback path for subclassed layers
            for value in vars(obj).values():
                yield from visit_container(value)

    def visit_container(value):
        if isinstance(value, layers.Layer):
            yield from visit(value)
        elif isinstance(value, (list, tuple, set)):
            for item in value:
                yield from visit_container(item)
        elif isinstance(value, dict):
            for item in value.values():
                yield from visit_container(item)

    yield from visit(root)


def resample_random_projection_filters(model):
    """
    Resample random-filter banks in every novel mixed conv layer.
    """
    for layer in _iter_nested_keras_layers(model):
        if isinstance(layer, DynamicBudgetedOrthogonalRPFAttentionConv2D):
            layer.resample_random_filters()


def inspect_dynamic_rpf_layers(model):
    rows = []
    for layer in _iter_nested_keras_layers(model):
        if isinstance(layer, DynamicBudgetedOrthogonalRPFAttentionConv2D):
            rows.append({
                #"layer": layer.name,
                "last_rp_ratio": float(layer.last_rp_ratio.numpy()),
                "last_num_random_filters": int(round(float(layer.last_num_random.numpy()))),
                "total_filters": int(layer.filters),
                "orthogonalized_random_bank": bool(layer.orthogonalize_random_bank),
            })
    return rows

In [ ]:
###### ORaSWAN
class DynamicBudgetedOrthogonalRPFAttentionConv2D(layers.Layer):
    """
    BCOP style Novel mixed convolution with a CERTIFIED orthogonal parameterization
    for the random-projection branch.

    """

    def __init__(self,
                 filters,
                 kernel_size,
                 strides=1,
                 padding="same",
                 use_bias=False,
                 activation=True,
                 noise_std=0.05,
                 trainable_kernel_l2=1e-2,
                 rp_ratio_min=0.10,
                 rp_ratio_max=0.90,
                 mask_temperature=25.0,
                 orthogonalize_random_bank=True,
                 rp_sigma_max=1.0,
                 att_sigma_max=1.0,
                 mix_sigma_max=1.5,
                 global_noise_factor=0.2,
                 feature_noise_factor=0.2,
                 learning_rate=0.001,
                 spatial_noise_factor=0.6,
                 channel_noise_factor=0.6,
                 att_num_layers=3,
                 use_eval_noise=True,
                 projector_rank_ratio=0.5,
                 **kwargs):
        super().__init__(**kwargs)

        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        if isinstance(strides, int):
            strides = (strides, strides)

        self.filters = int(filters)
        self.kernel_size = tuple(kernel_size)
        self.strides = tuple(strides)
        self.padding = str(padding).lower()
        self.use_bias = bool(use_bias)
        self.activation = bool(activation)

        self.noise_std = float(noise_std)
        self.trainable_kernel_l2 = float(trainable_kernel_l2)

        self.rp_ratio_min = float(rp_ratio_min)
        self.rp_ratio_max = float(rp_ratio_max)
        self.mask_temperature = float(mask_temperature)
        self.orthogonalize_random_bank = bool(orthogonalize_random_bank)

        self.rp_sigma_max = rp_sigma_max
        self.att_sigma_max = att_sigma_max
        self.mix_sigma_max = mix_sigma_max

        self.global_noise_factor = float(global_noise_factor)
        self.feature_noise_factor = float(feature_noise_factor)
        self.learning_rate = float(learning_rate)
        self.spatial_noise_factor = float(spatial_noise_factor)
        self.channel_noise_factor = float(channel_noise_factor)
        self.att_num_layers = int(att_num_layers)
        self.use_eval_noise = bool(use_eval_noise)
        self.projector_rank_ratio = float(projector_rank_ratio)

        if self.filters < 2:
            raise ValueError("filters must be >= 2.")

        if not (0.0 < self.rp_ratio_min < self.rp_ratio_max < 1.0):
            raise ValueError("Require 0 < rp_ratio_min < rp_ratio_max < 1.")

        if not (0.0 < self.projector_rank_ratio <= 1.0):
            raise ValueError("projector_rank_ratio must be in (0, 1].")

        self.bn = layers.BatchNormalization()
        self.act = layers.ReLU() if self.activation else None

        self.combined_noise = CombinedNoiseLayer(
            global_noise_factor=self.global_noise_factor,
            feature_noise_factor=self.feature_noise_factor,
            learning_rate=self.learning_rate,
        )

        self.budget_dense1 = layers.Dense(8, activation='relu')
        self.budget_dense2 = layers.Dense(1, activation=None)

        self.in_channels = None
        self.orig_in_channels = None
        self.flat_dim = None
        self.stride_adapter_active = False
        self.block_size = 1

        self.random_kernel = None
        self.att_kernel = None
        self.bias = None

        self.rho_logit = None
        self.theta_global_feat = None
        self.theta_att_feat = None

        self.last_rp_ratio = None
        self.last_num_random = None

    def _paper_random_stddev(self):
        kh, kw = self.kernel_size
        if kh == kw:
            return 1.0 / float(kh)
        return 1.0 / math.sqrt(float(kh * kw))

    def _random_orthogonal_matrix(self, n, dtype):
        a = tf.random.normal([n, n], dtype=dtype)
        q, r = tf.linalg.qr(a, full_matrices=True)
        d = tf.sign(tf.linalg.diag_part(r))
        d = tf.where(tf.equal(d, 0), tf.ones_like(d), d)
        q = q * d[tf.newaxis, :]
        return q

    def _symmetric_projection(self, n, dtype):
        rank = max(1, int(round(float(n) * self.projector_rank_ratio)))
        q = self._random_orthogonal_matrix(n, dtype)
        c = q[:, :rank]
        return tf.matmul(c, c, transpose_b=True)

    def _block_orth(self, p1, p2):
        eye = tf.eye(tf.shape(p1)[0], dtype=p1.dtype)
        return [
            [tf.matmul(p1, p2), tf.matmul(p1, eye - p2)],
            [tf.matmul(eye - p1, p2), tf.matmul(eye - p1, eye - p2)],
        ]

    def _matrix_conv(self, m1, m2):
        k = len(m1)
        l = len(m2)
        size = k + l - 1
        zero = tf.zeros_like(m1[0][0])
        result = [[zero for _ in range(size)] for _ in range(size)]
        for i in range(size):
            for j in range(size):
                acc = tf.zeros_like(zero)
                for index1 in range(min(k, i + 1)):
                    for index2 in range(min(k, j + 1)):
                        if (i - index1) < l and (j - index2) < l:
                            acc = acc + tf.matmul(
                                m1[index1][index2],
                                m2[i - index1][j - index2]
                            )
                result[i][j] = acc
        return result

    def _generate_bcop_random_kernel(self, dtype):
        kh, kw = self.kernel_size
        if kh != kw:
            raise ValueError("BCOP random bank implementation below assumes square kernels.")
        ksize = kh

        orig_cin = int(self.in_channels)
        orig_cout = int(self.filters)

        flipped = False
        cin = orig_cin
        cout = orig_cout

        if cin > cout:
            flipped = True
            cin, cout = cout, cin

        H_full = self._random_orthogonal_matrix(cout, dtype)
        H = H_full[:cin, :]   # [cin, cout]

        if ksize == 1:
            kernel = tf.reshape(H, [1, 1, cin, cout])
        else:
            p = self._block_orth(
                self._symmetric_projection(cout, dtype),
                self._symmetric_projection(cout, dtype)
            )
            for _ in range(1, ksize - 1):
                p = self._matrix_conv(
                    p,
                    self._block_orth(
                        self._symmetric_projection(cout, dtype),
                        self._symmetric_projection(cout, dtype)
                    )
                )

            blocks = []
            for i in range(ksize):
                row = []
                for j in range(ksize):
                    row.append(tf.matmul(H, p[i][j]))   # [cin, cout]
                blocks.append(row)

            kernel = tf.stack(
                [tf.stack(row, axis=0) for row in blocks],
                axis=0
            )  # [k, k, cin, cout]

        if flipped:
            kernel = tf.transpose(kernel, [0, 1, 3, 2])  # -> [k, k, orig_cin, orig_cout]

        if self.rp_sigma_max is not None:
            kernel = kernel * tf.cast(self.rp_sigma_max, dtype)

        return kernel

    def build(self, input_shape):
        kh, kw = self.kernel_size

        self.orig_in_channels = int(input_shape[-1])

        # BCOP and related exact orthogonal-convolution constructions are
        # naturally stride-1. For stride > 1 we use an invertible downsampling
        # adapter so the certified random bank still operates as a stride-1 conv.
        self.stride_adapter_active = (
            self.orthogonalize_random_bank and
            (self.strides != (1, 1))
        )

        if self.stride_adapter_active:
            if self.strides[0] != self.strides[1]:
                raise ValueError(
                    "Certified BCOP random bank currently requires equal spatial strides."
                )
            self.block_size = int(self.strides[0])
            self.in_channels = int(self.orig_in_channels * self.block_size * self.block_size)
        else:
            self.block_size = 1
            self.in_channels = int(self.orig_in_channels)

        self.flat_dim = int(kh * kw * self.in_channels)

        self.random_kernel = self.add_weight(
            shape=(kh, kw, self.in_channels, self.filters),
            initializer='zeros',
            trainable=False
        )

        self.att_kernel = self.add_weight(
            shape=(kh, kw, self.in_channels, self.filters),
            initializer='he_normal',
            regularizer=tf.keras.regularizers.l2(self.trainable_kernel_l2),
            trainable=True
        )

        if self.use_bias:
            self.bias = self.add_weight(
                shape=(self.filters,),
                initializer='zeros',
                trainable=True
            )

        self.rho_logit = self.add_weight(
            shape=(),
            initializer=tf.keras.initializers.Constant(-2.0),
            trainable=True
        )
        self.theta_global_feat = self.add_weight(
            shape=(),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )
        self.theta_att_feat = self.add_weight(
            shape=(),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )

        self.last_rp_ratio = self.add_weight(
            shape=(),
            initializer='zeros',
            trainable=False
        )
        self.last_num_random = self.add_weight(
            shape=(),
            initializer='zeros',
            trainable=False
        )

        super().build(input_shape)
        self.resample_random_filters()

    ## ORaFil
    def resample_random_filters(self):
        if self.orthogonalize_random_bank:
            sampled = self._generate_bcop_random_kernel(self.random_kernel.dtype)
        else:
            sampled = tf.random.normal(
                shape=self.random_kernel.shape,
                mean=0.0,
                stddev=self._paper_random_stddev(),
                dtype=self.random_kernel.dtype
            )
        self.random_kernel.assign(sampled)

    def _bool_training(self, training):
        return True if training is True else False

    def _compute_dynamic_ratio(self, x, delta_gf, delta_att):
        feat_abs_mean = tf.reduce_mean(tf.abs(x))
        feat_std = tf.math.reduce_std(x)
        gf_delta_mag = tf.reduce_mean(tf.abs(delta_gf))
        att_delta_mag = tf.reduce_mean(tf.abs(delta_att))

        controller_input = tf.reshape(
            tf.stack([feat_abs_mean, feat_std, gf_delta_mag, att_delta_mag]),
            [1, 4]
        )
        h = self.budget_dense1(controller_input)
        logit = self.budget_dense2(h)[0, 0]
        ratio = self.rp_ratio_min + (self.rp_ratio_max - self.rp_ratio_min) * tf.sigmoid(logit)
        return ratio

    def _build_straight_through_mask(self, ratio, dtype):
        positions = tf.cast(tf.range(self.filters), dtype) + tf.cast(0.5, dtype)
        k_float = ratio * tf.cast(self.filters, dtype)

        soft_mask = tf.sigmoid(
            tf.cast(self.mask_temperature, dtype) * (k_float - positions)
        )
        hard_mask = tf.cast(positions < k_float, dtype)

        mask = soft_mask + tf.stop_gradient(hard_mask - soft_mask)
        num_random = tf.reduce_sum(hard_mask)
        return tf.reshape(mask, [1, 1, 1, self.filters]), num_random

    def _prepare_input(self, x):
        if self.stride_adapter_active:
            x = tf.nn.space_to_depth(x, block_size=self.block_size)
            strides_tf = [1, 1, 1, 1]
        else:
            strides_tf = [1, self.strides[0], self.strides[1], 1]
        return x, strides_tf

    def _circular_same_pad(self, x):
        kh, kw = self.kernel_size
        pad_h = kh // 2
        pad_w = kw // 2

        if pad_h > 0:
            x = tf.concat([x[:, -pad_h:, :, :], x, x[:, :pad_h, :, :]], axis=1)
        if pad_w > 0:
            x = tf.concat([x[:, :, -pad_w:, :], x, x[:, :, :pad_w, :]], axis=2)
        return x

    ## SWANFil
    def _build_attention_noisy_kernel(self, x, training=None):
        training_flag = self._bool_training(training)

        if training_flag or self.use_eval_noise:
            x_gf = self.combined_noise(x, training=True)
        else:
            x_gf = self.combined_noise(x, training=False)
        delta_gf = x_gf - x

        if self.in_channels < 8:
            delta_att = tf.zeros_like(delta_gf)
        else:
            delta_att = x_gf + x

        w_g = tf.nn.softplus(self.theta_global_feat)
        w_a = tf.nn.softplus(self.theta_att_feat)
        x_fused = (w_g * delta_gf + w_a * delta_att) / (w_g + w_a + 1e-6)

        gate_in = tf.reduce_mean(x_fused, axis=[0, 1, 2])
        gate_in = gate_in / (tf.reduce_max(tf.abs(gate_in)) + 1e-6)
        gate_in = tf.reshape(gate_in, [1, 1, self.in_channels, 1])

        eps = tf.random.normal(
            shape=tf.shape(self.att_kernel),
            mean=0.0,
            stddev=self.noise_std,
            dtype=self.att_kernel.dtype
        )
        noise_gated = eps * gate_in
        rho = tf.nn.softplus(self.rho_logit)

        W_att_noisy = self.att_kernel + rho * noise_gated

        if (not self.orthogonalize_random_bank) and (self.att_sigma_max is not None):
            W_att_noisy = spectral_clip_flat_kernel(W_att_noisy, self.att_sigma_max)

        return W_att_noisy, delta_gf, delta_att

    def call(self, x, training=None):
        x_prepared, strides_tf = self._prepare_input(x)

        ### SWANFil
        W_att_noisy, delta_gf, delta_att = self._build_attention_noisy_kernel(
            x_prepared, training=training
        )

        ratio = self._compute_dynamic_ratio(x_prepared, delta_gf, delta_att)
        mask, num_random = self._build_straight_through_mask(ratio, x_prepared.dtype)

        self.last_rp_ratio.assign(tf.cast(ratio, self.last_rp_ratio.dtype))
        self.last_num_random.assign(tf.cast(num_random, self.last_num_random.dtype))

        # Mixed filter bank: some channels come from the certified BCOP-style
        # random branch; the rest come from the attention-noisy branch.
        W_mix = mask * self.random_kernel + (1.0 - mask) * W_att_noisy

        if (not self.orthogonalize_random_bank) and (self.mix_sigma_max is not None):
            W_mix = spectral_clip_flat_kernel(W_mix, self.mix_sigma_max)

        # For the BCOP random bank, use circular padding for the actual conv
        # to preserve the orthogonal-convolution setting.
        if self.orthogonalize_random_bank and self.padding == "same":
            x_conv = self._circular_same_pad(x_prepared)
            padding_tf = "VALID"
        else:
            x_conv = x_prepared
            padding_tf = self.padding.upper()

        y = tf.nn.conv2d(x_conv, W_mix, strides=strides_tf, padding=padding_tf)

        if self.bias is not None:
            y = tf.nn.bias_add(y, self.bias)

        y = self.bn(y, training=training)
        if self.act is not None:
            y = self.act(y)
        return y

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "strides": self.strides,
            "padding": self.padding,
            "use_bias": self.use_bias,
            "activation": self.activation,
            "noise_std": self.noise_std,
            "trainable_kernel_l2": self.trainable_kernel_l2,
            "rp_ratio_min": self.rp_ratio_min,
            "rp_ratio_max": self.rp_ratio_max,
            "mask_temperature": self.mask_temperature,
            "orthogonalize_random_bank": self.orthogonalize_random_bank,
            "rp_sigma_max": self.rp_sigma_max,
            "att_sigma_max": self.att_sigma_max,
            "mix_sigma_max": self.mix_sigma_max,
            "global_noise_factor": self.global_noise_factor,
            "feature_noise_factor": self.feature_noise_factor,
            "learning_rate": self.learning_rate,
            "spatial_noise_factor": self.spatial_noise_factor,
            "channel_noise_factor": self.channel_noise_factor,
            "att_num_layers": self.att_num_layers,
            "use_eval_noise": self.use_eval_noise,
            "projector_rank_ratio": self.projector_rank_ratio,
        })
        return config


In [ ]:

# ============================================================
# 4. ResNet-18 with the novel mixed conv
# ============================================================

class NovelResidualBlock(layers.Layer):
    def __init__(self,
                 filters,
                 stride=1,
                 noise_std=0.05,
                 trainable_kernel_l2=1e-3,
                 rp_ratio_min=0.10,
                 rp_ratio_max=0.90,
                 mask_temperature=25.0,
                 orthogonalize_random_bank=True,
                 rp_sigma_max=1.0,
                 att_sigma_max=1.0,
                 mix_sigma_max=1.5,
                 spatial_noise_factor=0.6,
                 channel_noise_factor=0.6,
                 att_num_layers=3,
                 use_eval_noise=True,
                 use_sani_tail=True,
                 **kwargs):
        super().__init__(**kwargs)
        self.filters = int(filters)
        self.stride = int(stride)
        self.use_sani_tail = bool(use_sani_tail)

        ## ORaFil
        self.conv1 = DynamicBudgetedOrthogonalRPFAttentionConv2D(
            filters=self.filters,
            kernel_size=3,
            strides=self.stride,
            padding="same",
            use_bias=False,
            activation=True,
            noise_std=noise_std,
            trainable_kernel_l2=trainable_kernel_l2,
            rp_ratio_min=rp_ratio_min,
            rp_ratio_max=rp_ratio_max,
            mask_temperature=mask_temperature,
            orthogonalize_random_bank=orthogonalize_random_bank,
            rp_sigma_max=rp_sigma_max,
            att_sigma_max=att_sigma_max,
            mix_sigma_max=mix_sigma_max,
            spatial_noise_factor=spatial_noise_factor,
            channel_noise_factor=channel_noise_factor,
            att_num_layers=att_num_layers,
            use_eval_noise=use_eval_noise,
            #name=self.name + "_conv1"
        )

        ## ORaFil
        self.conv2 = DynamicBudgetedOrthogonalRPFAttentionConv2D(
            filters=self.filters,
            kernel_size=3,
            strides=1,
            padding="same",
            use_bias=False,
            activation=False,
            noise_std=noise_std,
            trainable_kernel_l2=trainable_kernel_l2,
            rp_ratio_min=rp_ratio_min,
            rp_ratio_max=rp_ratio_max,
            mask_temperature=mask_temperature,
            orthogonalize_random_bank=orthogonalize_random_bank,
            rp_sigma_max=rp_sigma_max,
            att_sigma_max=att_sigma_max,
            mix_sigma_max=mix_sigma_max,
            spatial_noise_factor=spatial_noise_factor,
            channel_noise_factor=channel_noise_factor,
            att_num_layers=att_num_layers,
            use_eval_noise=use_eval_noise,
            #name=self.name + "_conv2"
        )

        self.proj = None
        self.proj_bn = None

        ### 
        if self.use_sani_tail:
            self.sani_tail = SoftplusMergedAttentionNoiseLayer(
                global_noise_factor=0.2,
                feature_noise_factor=0.2,
                learning_rate=0.001,
                spatial_noise_factor=spatial_noise_factor,
                channel_noise_factor=channel_noise_factor,
                num_layers=att_num_layers,
                use_eval_noise=use_eval_noise,
                #name=self.name + "_sani"
            )
        else:
            self.sani_tail = None

    def build(self, input_shape):
        in_ch = int(input_shape[-1])
        if in_ch != self.filters or self.stride != 1:
            self.proj = Conv2D(
                filters=self.filters,
                kernel_size=1,
                strides=self.stride,
                padding="same",
                use_bias=False,
                kernel_initializer="he_normal",
                #name=self.name + "_proj_conv"
            )
            self.proj_bn = layers.BatchNormalization()
        super().build(input_shape)

    def call(self, x, training=None):
        training_flag = True if training is True else False

        y = self.conv1(x, training=training_flag)
        y = self.conv2(y, training=training_flag)

        if self.proj is not None:
            shortcut = self.proj_bn(self.proj(x), training=training_flag)
        else:
            shortcut = x

        out = tf.nn.relu(y + shortcut)

        '''if self.sani_tail is not None:
            out = self.sani_tail(out, training=training_flag)'''
        return out


def _make_stage_novel(x,
                      filters,
                      num_blocks,
                      stride,
                      noise_std,
                      trainable_kernel_l2,
                      rp_ratio_min,
                      rp_ratio_max,
                      mask_temperature,
                      orthogonalize_random_bank,
                      rp_sigma_max,
                      att_sigma_max,
                      mix_sigma_max,
                      spatial_noise_factor,
                      channel_noise_factor,
                      att_num_layers,
                      use_eval_noise,
                      name_prefix):
    x = NovelResidualBlock(
        filters=filters,
        stride=stride,
        noise_std=noise_std,
        trainable_kernel_l2=trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        #name=name_prefix + "_block0"
    )(x)

    for i in range(1, num_blocks):
        x = NovelResidualBlock(
            filters=filters,
            stride=1,
            noise_std=noise_std,
            trainable_kernel_l2=trainable_kernel_l2,
            rp_ratio_min=rp_ratio_min,
            rp_ratio_max=rp_ratio_max,
            mask_temperature=mask_temperature,
            orthogonalize_random_bank=orthogonalize_random_bank,
            rp_sigma_max=rp_sigma_max,
            att_sigma_max=att_sigma_max,
            mix_sigma_max=mix_sigma_max,
            spatial_noise_factor=spatial_noise_factor,
            channel_noise_factor=channel_noise_factor,
            att_num_layers=att_num_layers,
            use_eval_noise=use_eval_noise,
            #name=f"{name_prefix}_block{i}"
        )(x)

    return x


def build_resnet18_novel_dynamic_rpf(input_shape=(128, 128, 3),
                                     num_classes=5,
                                     base_width=64,
                                     stem_use_novel=True,
                                     noise_std=0.05,
                                     stem_trainable_kernel_l2=1e-2,
                                     block_trainable_kernel_l2=1e-3,
                                     rp_ratio_min=0.10,
                                     rp_ratio_max=0.90,
                                     mask_temperature=25.0,
                                     orthogonalize_random_bank=True,
                                     rp_sigma_max=1.0,
                                     att_sigma_max=1.0,
                                     mix_sigma_max=1.5,
                                     spatial_noise_factor=0.6,
                                     channel_noise_factor=0.6,
                                     att_num_layers=3,
                                     use_eval_noise=True,
                                     output_logits=True):
    inputs = tf.keras.Input(shape=input_shape)

    if stem_use_novel:
        x = DynamicBudgetedOrthogonalRPFAttentionConv2D(
            filters=base_width,
            kernel_size=3,
            strides=1,
            padding="same",
            use_bias=False,
            activation=True,
            noise_std=noise_std,
            trainable_kernel_l2=stem_trainable_kernel_l2,
            rp_ratio_min=rp_ratio_min,
            rp_ratio_max=rp_ratio_max,
            mask_temperature=mask_temperature,
            orthogonalize_random_bank=orthogonalize_random_bank,
            rp_sigma_max=rp_sigma_max,
            att_sigma_max=att_sigma_max,
            mix_sigma_max=mix_sigma_max,
            spatial_noise_factor=spatial_noise_factor,
            channel_noise_factor=channel_noise_factor,
            att_num_layers=att_num_layers,
            use_eval_noise=use_eval_noise,
            #name="stem_novel"
        )(inputs)
    else:
        # Default stem kept standard so your unchanged channel-attention path
        # is fully active from stage 1 onward.
        x = Conv2D(
            filters=base_width,
            kernel_size=3,
            strides=1,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            #name="stem_conv"
        )(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)

    x = _make_stage_novel(
        x, base_width, num_blocks=2, stride=1,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage1"
    )

    x = _make_stage_novel(
        x, base_width * 2, num_blocks=2, stride=2,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage2"
    )

    x = _make_stage_novel(
        x, base_width * 4, num_blocks=2, stride=2,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage3"
    )

    x = _make_stage_novel(
        x, base_width * 8, num_blocks=2, stride=2,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage4"
    )

    x = layers.GlobalAveragePooling2D(name="avgpool")(x)
    outputs = layers.Dense(
        num_classes,
        activation=None if output_logits else "softmax",
        #name="head_dense"
    )(x)

    return tf.keras.Model(inputs, outputs)


In [ ]:
## SWAN
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense, Add

class TrainableSpatialAttentionLayer1(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer1, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  trainable=True)
        self.concat = Add()
        super(TrainableSpatialAttentionLayer1, self).build(input_shape)

    def call(self, inputs):
        inputs1, inputs2 = inputs
        attention_weights1 = self.convolution(inputs1)
        attention_weights2 = self.convolution(inputs2)
        attention_weights = self.concat([attention_weights1, attention_weights2])
        con_inputs = self.concat([inputs1, inputs2])
        return tf.multiply(con_inputs, attention_weights)

class TrainableChannelAttentionLayer1(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer1, self).__init__(**kwargs)

    def build(self, input_shape):
        inputs1, inputs2 = input_shape
        num_channels = inputs1[-1]
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.concat = Add()
        self.dense1 = Dense(units=inputs1[-1] // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=inputs1[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer1, self).build(input_shape)

    def call(self, inputs):
        inputs1, inputs2 = inputs
        avg_pool1 = self.global_avg_pooling(inputs1)
        avg_pool2 = self.global_avg_pooling(inputs2)
        avg_pool = self.concat([avg_pool1, avg_pool2])
        
        dense1_out = self.dense1(avg_pool)
        
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)

        con_inputs = self.concat([inputs1, inputs2])
        #return tf.multiply(coninputs, attention_weights)

        return tf.multiply(con_inputs, channel_attention_weights)

'''class TrainableCombinedAttentionLayer1(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer1, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        inputs1, inputs2 = inputs
        spatial_attention_output = inputs1
        channel_attention_output = inputs1

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, 
                                         stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, 
                                         stddev=self.channel_noise_factor)

        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        channel_noise *= self.channel_noise_weight + (channel_noise * self.channel_noise_weight)

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        return tf.multiply(inputs1, combined_attention)
'''

class TrainableCombinedAttentionLayer1(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, **kwargs):
        super(TrainableCombinedAttentionLayer1, self).__init__(**kwargs)
        
        # Single attention layer for both spatial and channel attention
        self.spatial_attention = TrainableSpatialAttentionLayer1()
        self.channel_attention = TrainableChannelAttentionLayer1()
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor
        self.concat = Add()

    def call(self, inputs):
        inputs1, inputs2 = inputs
        
        # Apply spatial attention
        spatial_attention_output = self.spatial_attention([inputs1, inputs2])
        
        # Apply channel attention
        channel_attention_output = self.channel_attention([inputs1, inputs2])

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), 
                                         mean=0, stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), 
                                         mean=0, stddev=self.channel_noise_factor)
        
        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        channel_noise *= self.channel_noise_weight + (channel_noise * self.channel_noise_weight)

        # Scale the noise with trainable weights
        #spatial_noise *= self.spatial_noise_weight
        #channel_noise *= self.channel_noise_weight

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        '''# Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        '''
        
        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        con_inputs = self.concat([inputs1, inputs2])
        
        return tf.multiply(con_inputs, combined_attention)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense, Add


class TrainableSpatialAttentionLayer1(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer1, self).__init__(**kwargs)
        self.convolution = Conv2D(
            filters=1,
            kernel_size=(1, 1),
            activation='sigmoid',
            padding='same',
            trainable=True
        )
        self.concat = Add()

    def build(self, input_shape):
        super(TrainableSpatialAttentionLayer1, self).build(input_shape)

    def call(self, inputs):
        if not isinstance(inputs, (list, tuple)) or len(inputs) != 2:
            raise ValueError(
                "TrainableSpatialAttentionLayer1 expects [inputs1, inputs2]."
            )

        inputs1, inputs2 = inputs
        attention_weights1 = self.convolution(inputs1)
        attention_weights2 = self.convolution(inputs2)
        attention_weights = self.concat([attention_weights1, attention_weights2])
        con_inputs = self.concat([inputs1, inputs2])

        return tf.multiply(con_inputs, attention_weights)


class TrainableChannelAttentionLayer1(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer1, self).__init__(**kwargs)
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.concat = Add()
        self.dense1 = None
        self.dense2 = None

    def build(self, input_shape):
        if not isinstance(input_shape, (list, tuple)) or len(input_shape) != 2:
            raise ValueError(
                "TrainableChannelAttentionLayer1 expects two input shapes."
            )

        inputs1, inputs2 = input_shape
        num_channels = inputs1[-1]

        if num_channels is None:
            raise ValueError(
                "The channel dimension must be defined for TrainableChannelAttentionLayer1."
            )

        if num_channels < 2:
            raise ValueError(
                "The channel dimension must be at least 2 because dense1 uses num_channels // 2."
            )

        self.dense1 = Dense(units=num_channels // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=num_channels, activation='sigmoid', trainable=True)

        super(TrainableChannelAttentionLayer1, self).build(input_shape)

    def call(self, inputs):
        if not isinstance(inputs, (list, tuple)) or len(inputs) != 2:
            raise ValueError(
                "TrainableChannelAttentionLayer1 expects [inputs1, inputs2]."
            )

        inputs1, inputs2 = inputs
        avg_pool1 = self.global_avg_pooling(inputs1)
        avg_pool2 = self.global_avg_pooling(inputs2)
        avg_pool = self.concat([avg_pool1, avg_pool2])

        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(
            tf.expand_dims(channel_attention_weights, 1), 1
        )

        con_inputs = self.concat([inputs1, inputs2])

        return tf.multiply(con_inputs, channel_attention_weights)


class TrainableCombinedAttentionLayer1(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, **kwargs):
        super(TrainableCombinedAttentionLayer1, self).__init__(**kwargs)

        self.spatial_attention = TrainableSpatialAttentionLayer1()
        self.channel_attention = TrainableChannelAttentionLayer1()

        self.spatial_noise_weight = self.add_weight(
            shape=(1,),
            initializer='ones',
            trainable=True,
            name='spatial_noise_weight'
        )
        self.channel_noise_weight = self.add_weight(
            shape=(1,),
            initializer='ones',
            trainable=True,
            name='channel_noise_weight'
        )

        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor
        self.concat = Add()

    def call(self, inputs):
        if not isinstance(inputs, (list, tuple)) or len(inputs) != 2:
            raise ValueError(
                "TrainableCombinedAttentionLayer1 expects [inputs1, inputs2]."
            )

        inputs1, inputs2 = inputs

        spatial_attention_output = self.spatial_attention([inputs1, inputs2])
        channel_attention_output = self.channel_attention([inputs1, inputs2])

        spatial_noise = tf.random.normal(
            shape=tf.shape(spatial_attention_output),
            mean=0.0,
            stddev=self.spatial_noise_factor,
            dtype=spatial_attention_output.dtype
        )
        channel_noise = tf.random.normal(
            shape=tf.shape(channel_attention_output),
            mean=0.0,
            stddev=self.channel_noise_factor,
            dtype=channel_attention_output.dtype
        )

        spatial_noise = spatial_noise * (
            self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        )
        channel_noise = channel_noise * (
            self.channel_noise_weight + (channel_noise * self.channel_noise_weight)
        )

        spatial_attention_output = spatial_attention_output + spatial_noise
        channel_attention_output = channel_attention_output + channel_noise

        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        combined_attention1 = tf.multiply(
            spatial_attention_output, channel_attention_output
        )
        combined_attention2 = tf.multiply(
            spatial_attention_output, channel_attention_output
        )
        combined_attention3 = tf.multiply(
            spatial_attention_output, channel_attention_output
        )
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)

        con_inputs = self.concat([inputs1, inputs2])

        return tf.multiply(con_inputs, combined_attention)

In [ ]:
def build_resnet18_novel_dynamic_rpf(input_shape1=(128, 128, 3), input_shape2=(128, 128, 3),
                                     num_classes=5
                                     base_width=64,
                                     stem_use_novel=True,
                                     noise_std=0.05,
                                     stem_trainable_kernel_l2=1e-2,
                                     block_trainable_kernel_l2=1e-3,
                                     rp_ratio_min=0.10,
                                     rp_ratio_max=0.90,
                                     mask_temperature=25.0,
                                     orthogonalize_random_bank=True,
                                     rp_sigma_max=1.0,
                                     att_sigma_max=1.0,
                                     mix_sigma_max=1.5,
                                     spatial_noise_factor=0.6,
                                     channel_noise_factor=0.6,
                                     att_num_layers=3,
                                     use_eval_noise=True,
                                     output_logits=True):
    inputs1 = tf.keras.Input(shape=input_shape1)
    inputs2 = tf.keras.Input(shape=input_shape2)

    #if stem_use_novel:
    x1 = DynamicBudgetedOrthogonalRPFAttentionConv2D(
        filters=base_width,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        activation=True,
        noise_std=noise_std,
        trainable_kernel_l2=stem_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        #name="stem_novel"
    )(inputs1)

    x2 = DynamicBudgetedOrthogonalRPFAttentionConv2D(
        filters=base_width,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        activation=True,
        noise_std=noise_std,
        trainable_kernel_l2=stem_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        #name="stem_novel"
    )(inputs2)
    '''else:
        # Default stem kept standard so your unchanged channel-attention path
        # is fully active from stage 1 onward.
        x = Conv2D(
            filters=base_width,
            kernel_size=3,
            strides=1,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name="stem_conv"
        )(inputs1)'''

    x1 = layers.BatchNormalization()(x1)
    x1 = layers.ReLU()(x1)
    x2 = layers.BatchNormalization()(x2)
    x2 = layers.ReLU()(x2)

    x1 = _make_stage_novel(
        x1, base_width, num_blocks=2, stride=1,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage1_1"
    )

    x2 = _make_stage_novel(
        x2, base_width, num_blocks=2, stride=1,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage1_2"
    )

    att_after_res1 = TrainableCombinedAttentionLayer1(name="att_after_res1")
    x = att_after_res1([x1, x2])
    
    #x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                       # )([x1, x2])

    
    
    x1 = _make_stage_novel(
        x, base_width * 2, num_blocks=2, stride=2,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage2_1"
    )

    x2 = _make_stage_novel(
        x, base_width * 2, num_blocks=2, stride=2,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage2_1"
    )

    att_after_res2 = TrainableCombinedAttentionLayer1(name="att_after_res2")
    x = att_after_res2([x1, x2])
    
    x1 = _make_stage_novel(
        x, base_width * 4, num_blocks=2, stride=2,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage3_1"
    )

    x2 = _make_stage_novel(
        x, base_width * 4, num_blocks=2, stride=2,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage3_2"
    )

    att_after_res3 = TrainableCombinedAttentionLayer1(name="att_after_res3")
    x = att_after_res3([x1, x2])
    
    x1 = _make_stage_novel(
        x, base_width * 8, num_blocks=2, stride=2,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage4_1"
    )

    x2 = _make_stage_novel(
        x, base_width * 8, num_blocks=2, stride=2,
        noise_std=noise_std,
        trainable_kernel_l2=block_trainable_kernel_l2,
        rp_ratio_min=rp_ratio_min,
        rp_ratio_max=rp_ratio_max,
        mask_temperature=mask_temperature,
        orthogonalize_random_bank=orthogonalize_random_bank,
        rp_sigma_max=rp_sigma_max,
        att_sigma_max=att_sigma_max,
        mix_sigma_max=mix_sigma_max,
        spatial_noise_factor=spatial_noise_factor,
        channel_noise_factor=channel_noise_factor,
        att_num_layers=att_num_layers,
        use_eval_noise=use_eval_noise,
        name_prefix="stage4_2"
    )

    
    #x = tf.keras.layers.Concatenate(axis=-1)([x1, x2])

    att_after_res4 = TrainableCombinedAttentionLayer1(name="att_after_res4")
    x = att_after_res4([x1, x2])

    print(x.shape)
    
    #x = tf.keras.layers.concat([x1, x2], axis=1)
    #x = x1+x2
    
    x = layers.GlobalAveragePooling2D()(x)
    
    outputs = layers.Dense(
        num_classes,
        activation = "softmax",
        #name="head_dense1"
    )(x)

    

    return tf.keras.Model([inputs1, inputs2], [outputs])


In [ ]:

# ============================================================
# 6. Build model
# ============================================================

model = build_resnet18_novel_dynamic_rpf(
    input_shape1=(128, 128, 3), input_shape2=(128, 128, 3),
    num_classes1=5,     num_classes2=7,
    base_width=64,
    stem_use_novel=True,          # set True if you want the novel layer in the stem too
    noise_std=0.05,
    stem_trainable_kernel_l2=1e-2,
    block_trainable_kernel_l2=1e-3,
    rp_ratio_min=0.10,
    rp_ratio_max=0.90,
    mask_temperature=25.0,
    orthogonalize_random_bank=True,
    rp_sigma_max=1.0,
    att_sigma_max=1.0,
    mix_sigma_max=1.5,
    spatial_noise_factor=0.6,
    channel_noise_factor=0.6,
    att_num_layers=3,
    use_eval_noise=True,
    output_logits=True
)


optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss=['categorical_crossentropy'],
    metrics=['accuracy']
)

In [ ]:
model.summary()

In [ ]:

# ============================================================
# 8. Callbacks and fit
# ============================================================


batch_size = 8
epochs = 100

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath="best_novel_dynamic_rpf_resnet18.keras",
    save_weights_only=False,
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=60,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.1,
    patience=30,
    verbose=1,
    min_lr=1e-5
)

callbacks = [checkpoint_callback, early_stopping, reduce_lr]

history = model.fit(
    [X_train_s, X_train_h], [y_train_s, y_train_h],
    validation_data=([X_val_s, X_val_h1], [y_val_s, y_val_h1]),
    epochs=epochs,
    verbose=1,
    callbacks=callbacks,
    shuffle=True, 
    batch_size = 8
)


In [ ]:
# ============================================================
# Randomized smoothing for a multimodal fusion Keras model
# Cohen et al. certification with sigma and radius grids
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import beta, norm


# ============================================================
# Configuration
# ============================================================

ABSTAIN = -1

SIGMA_GRID = [0.25, 0.50, 1.00]
RADIUS_GRID = np.round(np.arange(0.0, 2.0 + 1e-12, 0.25), 2)

N0_SELECT = 100          # samples for selecting the top class
N_CERTIFY = 10000        # certification samples; increase for final reporting if compute allows
ALPHA = 0.001            # failure probability
BATCH_SIZE = 64

# Keep None for standard Gaussian randomized smoothing.
# Using clipping changes the exact Gaussian distribution.
CLIP_VALUES = None       # Example alternative: (0.0, 1.0)

OUTPUT_INDEX = 0         # Use 0 if model output is [fused_output]


# ============================================================
# Utility functions
# ============================================================

def labels_to_int(y):
    """
    Convert labels to integer class ids.
    Supports sparse labels with shape [N] and one-hot labels with shape [N, C].
    """
    y = np.asarray(y)

    if y.ndim == 1:
        return y.astype(np.int64)

    if y.ndim == 2:
        return np.argmax(y, axis=1).astype(np.int64)

    raise ValueError(f"Unsupported label shape {y.shape}. Expected [N] or [N, C].")


def select_model_output(model_output, output_index=0):
    """
    Handles models returning:
      output
      [output]
      (output,)
      {"name": output}
    """
    if isinstance(model_output, dict):
        if isinstance(output_index, str):
            return model_output[output_index]
        return list(model_output.values())[output_index]

    if isinstance(model_output, (list, tuple)):
        return model_output[output_index]

    return model_output


def infer_num_classes(model=None, y=None, output_index=0):
    """
    Infer the number of output classes from labels and/or model output shape.
    """
    candidates = []

    if y is not None:
        y_arr = np.asarray(y)

        if y_arr.ndim == 2:
            candidates.append(int(y_arr.shape[1]))
        elif y_arr.ndim == 1:
            candidates.append(int(np.max(y_arr)) + 1)

    if model is not None:
        try:
            out_shape = model.output_shape

            if isinstance(out_shape, dict):
                if isinstance(output_index, str):
                    out_shape = out_shape[output_index]
                else:
                    out_shape = list(out_shape.values())[output_index]

            elif isinstance(out_shape, list):
                out_shape = out_shape[output_index]

            elif isinstance(out_shape, tuple):
                # Single-output shape is usually (None, C).
                # Multi-output shape may be ((None, C1), (None, C2)).
                if len(out_shape) > 0 and isinstance(out_shape[0], (tuple, list, tf.TensorShape)):
                    out_shape = out_shape[output_index]

            last_dim = int(out_shape[-1])
            candidates.append(last_dim)

        except Exception:
            pass

    if not candidates:
        raise ValueError(
            "Could not infer num_classes. Pass num_classes explicitly to the evaluation function."
        )

    return int(max(candidates))


def iter_keras_layers(root):
    """
    Recursively iterate through Keras layers and nested sublayers.
    """
    seen = set()

    def visit(layer):
        if not isinstance(layer, tf.keras.layers.Layer):
            return

        layer_id = id(layer)
        if layer_id in seen:
            return

        seen.add(layer_id)
        yield layer

        for child in getattr(layer, "layers", []) or []:
            yield from visit(child)

        for child in getattr(layer, "submodules", []) or []:
            if isinstance(child, tf.keras.layers.Layer):
                yield from visit(child)

    yield from visit(root)


def disable_custom_inference_noise(
    model,
    boolean_attributes=("use_eval_noise", "apply_noise_in_inference", "stochastic_inference"),
    float_attributes=(
        "noise_std",
        "eval_noise_std",
        "global_noise_factor",
        "feature_noise_factor",
        "spatial_noise_factor",
        "channel_noise_factor",
        "noise_factor",
    ),
    verbose=True,
):
    """
    Randomized smoothing certification assumes the base classifier is deterministic.

    This helper disables common custom-layer attributes that may introduce
    random noise during inference. It only changes attributes that already
    exist in the model's layers.
    """
    changed = []

    for layer in iter_keras_layers(model):
        for attr in boolean_attributes:
            if hasattr(layer, attr):
                old_value = getattr(layer, attr)
                try:
                    setattr(layer, attr, False)
                    changed.append((layer.name, attr, old_value, False))
                except Exception:
                    pass

        for attr in float_attributes:
            if hasattr(layer, attr):
                old_value = getattr(layer, attr)
                try:
                    setattr(layer, attr, 0.0)
                    changed.append((layer.name, attr, old_value, 0.0))
                except Exception:
                    pass

    if verbose:
        print(f"Updated {len(changed)} inference-noise attributes.")
        for item in changed[:20]:
            print("  layer=%s | %s: %s -> %s" % item)
        if len(changed) > 20:
            print(f"  ... plus {len(changed) - 20} more")

    return changed


def predict_classes_multimodal(model, x_batches, output_index=0):
    """
    Predict hard classes for a multimodal batch.

    Example:
        x_batches = [image_batch, ehr_batch]
    """
    x_tf = [tf.convert_to_tensor(x, dtype=tf.float32) for x in x_batches]

    output = model(x_tf, training=False)
    output = select_model_output(output, output_index=output_index)

    if tf.is_tensor(output):
        output = output.numpy()
    else:
        output = np.asarray(output)

    if output.ndim != 2:
        raise ValueError(f"Expected output shape [batch, classes], got {output.shape}")

    return np.argmax(output, axis=1).astype(np.int64)


def max_repeated_prediction_difference(model, x_modalities_batch, repeats=3, output_index=0):
    """
    Check whether the base classifier is deterministic at inference.

    x_modalities_batch should already include a batch dimension:
        [X_test_image[:4], X_test_ehr[:4]]
    """
    outputs = []
    x_tf = [tf.convert_to_tensor(x, dtype=tf.float32) for x in x_modalities_batch]

    for _ in range(repeats):
        output = model(x_tf, training=False)
        output = select_model_output(output, output_index=output_index)

        if tf.is_tensor(output):
            output = output.numpy()
        else:
            output = np.asarray(output)

        outputs.append(output.astype(np.float64))

    max_diff = 0.0

    for i in range(1, len(outputs)):
        max_diff = max(max_diff, float(np.max(np.abs(outputs[i] - outputs[0]))))

    return max_diff


# ============================================================
# Confidence bound and Gaussian sampling
# ============================================================

def lower_confidence_bound(k, n, alpha):
    """
    One-sided Clopper-Pearson lower confidence bound.

    k: number of times the selected class is predicted
    n: total Monte Carlo samples
    alpha: failure probability
    """
    k = int(k)
    n = int(n)

    if n <= 0:
        raise ValueError("n must be positive.")

    if k < 0 or k > n:
        raise ValueError("k must satisfy 0 <= k <= n.")

    if k == 0:
        return 0.0

    value = beta.ppf(alpha, k, n - k + 1)
    return float(np.clip(value, 0.0, 1.0 - 1e-15))


def get_clip_values_for_modality(clip_values, modality_index):
    """
    Supports:
      clip_values = None
      clip_values = (0.0, 1.0)
      clip_values = [(0.0, 1.0), (0.0, 1.0)]
    """
    if clip_values is None:
        return None

    if (
        isinstance(clip_values, (list, tuple))
        and len(clip_values) > 0
        and isinstance(clip_values[0], (list, tuple, np.ndarray))
    ):
        return clip_values[modality_index]

    return clip_values


def make_noisy_multimodal_batch(
    x_modalities,
    batch_size,
    sigma,
    rng,
    clip_values=None,
):
    """
    Create a noisy batch for one multimodal sample.

    The same scalar sigma is applied independently to every coordinate
    of every modality.
    """
    sigma = float(sigma)

    if sigma <= 0:
        raise ValueError("sigma must be positive.")

    noisy_batches = []

    for modality_index, x in enumerate(x_modalities):
        x = np.asarray(x, dtype=np.float32)

        x_batch = np.repeat(x[None, ...], batch_size, axis=0)

        noise = rng.normal(
            loc=0.0,
            scale=sigma,
            size=x_batch.shape
        ).astype(np.float32)

        x_noisy = x_batch + noise

        modality_clip = get_clip_values_for_modality(clip_values, modality_index)

        if modality_clip is not None:
            lo, hi = modality_clip
            x_noisy = np.clip(x_noisy, lo, hi)

        noisy_batches.append(x_noisy.astype(np.float32, copy=False))

    return noisy_batches


def sample_class_counts(
    model,
    x_modalities,
    sigma,
    num_samples,
    num_classes,
    batch_size=64,
    rng=None,
    clip_values=None,
    output_index=0,
):
    """
    Estimate class prediction counts under Gaussian noise.
    """
    if rng is None:
        rng = np.random.default_rng()

    counts = np.zeros(int(num_classes), dtype=np.int64)
    remaining = int(num_samples)

    while remaining > 0:
        current_batch_size = min(int(batch_size), remaining)

        noisy_batch = make_noisy_multimodal_batch(
            x_modalities=x_modalities,
            batch_size=current_batch_size,
            sigma=sigma,
            rng=rng,
            clip_values=clip_values,
        )

        predictions = predict_classes_multimodal(
            model=model,
            x_batches=noisy_batch,
            output_index=output_index,
        )

        batch_counts = np.bincount(predictions, minlength=int(num_classes))

        if len(batch_counts) != len(counts):
            raise ValueError(
                f"Prediction produced class index outside [0, {num_classes - 1}]. "
                f"Check num_classes. Got bincount length {len(batch_counts)}."
            )

        counts += batch_counts.astype(np.int64)
        remaining -= current_batch_size

    return counts


# ============================================================
# Certification for one sample
# ============================================================

def certify_one_sample(
    model,
    x_modalities,
    sigma,
    num_classes,
    n0=100,
    n=10000,
    alpha=0.001,
    batch_size=64,
    rng=None,
    clip_values=None,
    output_index=0,
):
    """
    Certify one multimodal sample.

    The certified radius is for the joint L2 norm over the concatenated
    multimodal input representation.
    """
    if rng is None:
        rng = np.random.default_rng()

    sigma = float(sigma)

    counts_selection = sample_class_counts(
        model=model,
        x_modalities=x_modalities,
        sigma=sigma,
        num_samples=n0,
        num_classes=num_classes,
        batch_size=batch_size,
        rng=rng,
        clip_values=clip_values,
        output_index=output_index,
    )

    selected_class = int(np.argmax(counts_selection))

    counts_certification = sample_class_counts(
        model=model,
        x_modalities=x_modalities,
        sigma=sigma,
        num_samples=n,
        num_classes=num_classes,
        batch_size=batch_size,
        rng=rng,
        clip_values=clip_values,
        output_index=output_index,
    )

    n_selected = int(counts_certification[selected_class])
    p_lower = lower_confidence_bound(n_selected, n, alpha)

    if p_lower <= 0.5:
        prediction = ABSTAIN
        radius = 0.0
        abstained = True
    else:
        prediction = selected_class
        radius = float(sigma * norm.ppf(p_lower))
        abstained = False

    return {
        "sigma": sigma,
        "prediction": int(prediction),
        "selected_class": int(selected_class),
        "abstained": bool(abstained),
        "certified_radius": float(radius),
        "p_lower": float(p_lower),
        "n_selected": int(n_selected),
        "n_certification": int(n),
        "n_selection": int(n0),
        "counts_selection": counts_selection.tolist(),
        "counts_certification": counts_certification.tolist(),
    }


# ============================================================
# Tables
# ============================================================

def build_robust_accuracy_table(certification_results, radius_grid):
    """
    Build robust accuracy table over the requested radius grid.

    Entry at radius r:
        mean(prediction is correct AND not abstained AND certified_radius >= r)

    Values are percentages.
    """
    rows = []

    for sigma, group in certification_results.groupby("sigma", sort=True):
        correct = group["correct"].to_numpy(dtype=bool)
        abstained = group["abstained"].to_numpy(dtype=bool)
        radii = group["certified_radius"].to_numpy(dtype=np.float64)

        row = {
            "sigma": float(sigma),
            "n_examples": int(len(group)),
        }

        for radius in radius_grid:
            robust_accuracy = np.mean(
                correct
                & (~abstained)
                & (radii >= float(radius))
            )

            row[f"r={float(radius):.2f}"] = 100.0 * float(robust_accuracy)

        rows.append(row)

    return pd.DataFrame(rows)


def build_sigma_summary_table(certification_results):
    """
    Build per-sigma summary table.
    """
    rows = []

    for sigma, group in certification_results.groupby("sigma", sort=True):
        correct = group["correct"].to_numpy(dtype=bool)
        abstained = group["abstained"].to_numpy(dtype=bool)
        radii = group["certified_radius"].to_numpy(dtype=np.float64)

        correct_non_abstained = correct & (~abstained)

        if np.any(correct_non_abstained):
            mean_radius_correct = float(np.mean(radii[correct_non_abstained]))
            median_radius_correct = float(np.median(radii[correct_non_abstained]))
        else:
            mean_radius_correct = 0.0
            median_radius_correct = 0.0

        rows.append({
            "sigma": float(sigma),
            "n_examples": int(len(group)),
            "smoothed_accuracy_r0_percent": 100.0 * float(np.mean(correct_non_abstained)),
            "abstain_rate_percent": 100.0 * float(np.mean(abstained)),
            "mean_certified_radius_all": float(np.mean(radii)),
            "median_certified_radius_all": float(np.median(radii)),
            "mean_certified_radius_correct_nonabstained": mean_radius_correct,
            "median_certified_radius_correct_nonabstained": median_radius_correct,
        })

    return pd.DataFrame(rows)


# ============================================================
# Full sigma-grid evaluation
# ============================================================

def evaluate_randomized_smoothing_grid(
    model,
    X_modalities,
    y,
    sigma_grid=(0.25, 0.50, 1.00),
    radius_grid=None,
    num_classes=None,
    n0=100,
    n=10000,
    alpha=0.001,
    batch_size=64,
    max_examples=None,
    indices=None,
    seed=12345,
    clip_values=None,
    output_index=0,
    progress_every=10,
):
    """
    Evaluate randomized smoothing over multiple sigma values.

    Parameters
    ----------
    model:
        Keras multimodal fusion model.

    X_modalities:
        List of input arrays, for example:
            [X_test_image, X_test_ehr]

    y:
        Sparse or one-hot labels.

    sigma_grid:
        Sigma values to evaluate separately.

    radius_grid:
        Radii used to report robust accuracy.

    Returns
    -------
    certification_results:
        Per-example certification results.

    robust_accuracy_table:
        Robust accuracy over the radius grid.

    sigma_summary_table:
        Summary statistics for each sigma.
    """
    if radius_grid is None:
        radius_grid = np.round(np.arange(0.0, 2.0 + 1e-12, 0.25), 2)

    X_modalities = [np.asarray(X) for X in X_modalities]
    y_int = labels_to_int(y)

    n_total = len(y_int)

    for modality_index, X in enumerate(X_modalities):
        if len(X) != n_total:
            raise ValueError(
                f"Modality {modality_index} has {len(X)} samples, "
                f"but labels have {n_total} samples."
            )

    if num_classes is None:
        num_classes = infer_num_classes(
            model=model,
            y=y,
            output_index=output_index,
        )

    num_classes = int(num_classes)

    if indices is None:
        indices = np.arange(n_total, dtype=np.int64)
    else:
        indices = np.asarray(indices, dtype=np.int64)

    if max_examples is not None:
        indices = indices[:int(max_examples)]

    all_rows = []

    for sigma_position, sigma in enumerate(sigma_grid):
        sigma = float(sigma)

        sigma_seed = None
        if seed is not None:
            sigma_seed = int(seed) + 1000003 * int(sigma_position)

        rng = np.random.default_rng(sigma_seed)

        print(
            f"\nCertifying sigma={sigma:.2f} | "
            f"examples={len(indices)} | n0={n0} | n={n}"
        )

        for local_position, sample_index in enumerate(indices):
            x_sample = [X[sample_index] for X in X_modalities]

            result = certify_one_sample(
                model=model,
                x_modalities=x_sample,
                sigma=sigma,
                num_classes=num_classes,
                n0=n0,
                n=n,
                alpha=alpha,
                batch_size=batch_size,
                rng=rng,
                clip_values=clip_values,
                output_index=output_index,
            )

            true_label = int(y_int[sample_index])
            prediction = int(result["prediction"])

            result.update({
                "index": int(sample_index),
                "true_label": true_label,
                "correct": bool(prediction == true_label),
            })

            all_rows.append(result)

            if progress_every is not None and progress_every > 0:
                finished = local_position + 1
                if finished % int(progress_every) == 0 or finished == len(indices):
                    print(f"sigma={sigma:.2f}: certified {finished}/{len(indices)}")

    certification_results = pd.DataFrame(all_rows)

    robust_accuracy_table = build_robust_accuracy_table(
        certification_results=certification_results,
        radius_grid=radius_grid,
    )

    sigma_summary_table = build_sigma_summary_table(
        certification_results=certification_results,
    )

    return certification_results, robust_accuracy_table, sigma_summary_table


# ============================================================
# Run section
# ============================================================
# This assumes your notebook already has:
#   model
#   X_test_image
#   X_test_ehr
#   y_test
#
# If your modality variable names are different, only change the list below:
#   X_modalities=[X_test_image, X_test_ehr]
# ============================================================

# Optional but recommended when custom layers add random noise during inference.
disable_custom_inference_noise(model, verbose=True)

# Verify deterministic inference.
determinism_gap = max_repeated_prediction_difference(
    model=model,
    x_modalities_batch=[X_test_image[:4], X_test_ehr[:4]],
    repeats=3,
    output_index=OUTPUT_INDEX,
)

print("Max repeated prediction difference:", determinism_gap)

if determinism_gap > 1e-5:
    raise RuntimeError(
        "The model is still stochastic during inference. "
        "Randomized smoothing certification requires a deterministic base classifier."
    )

# For a quick test, set max_examples to a small number such as 20.
# For full evaluation, keep max_examples=None.
certification_results, robust_accuracy_table, sigma_summary_table = evaluate_randomized_smoothing_grid(
    model=model,
    X_modalities=[X_test_image, X_test_ehr],
    y=y_test,
    sigma_grid=SIGMA_GRID,
    radius_grid=RADIUS_GRID,
    num_classes=None,
    n0=N0_SELECT,
    n=N_CERTIFY,
    alpha=ALPHA,
    batch_size=BATCH_SIZE,
    max_examples=None,
    indices=None,
    seed=12345,
    clip_values=CLIP_VALUES,
    output_index=OUTPUT_INDEX,
    progress_every=10,
)

print("\nRobust accuracy table over radius grid (%):")
print(robust_accuracy_table)

print("\nSigma summary table:")
print(sigma_summary_table)

# Optional CSV outputs
certification_results.to_csv("randomized_smoothing_certification_results.csv", index=False)
robust_accuracy_table.to_csv("randomized_smoothing_robust_accuracy_table.csv", index=False)
sigma_summary_table.to_csv("randomized_smoothing_sigma_summary.csv", index=False)

print("\nSaved CSV files:")
print("  randomized_smoothing_certification_results.csv")
print("  randomized_smoothing_robust_accuracy_table.csv")
print("  randomized_smoothing_sigma_summary.csv")